<a href="https://colab.research.google.com/github/deyarnab9862-create/python-projects/blob/python-dev/Email_Automation_Using_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!Pip install transformers
!pip install datasets
!pip install sentencepiece
!pip install accelerate
!pip install evaluate

/bin/bash: line 1: Pip: command not found
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from transformers import(
    AutoTokenizer,
    AutoModelForSeq2SeqLM,

)

import torch

In [ ]:
data = {
    "instruction":[
        "promotion request",
        "leave request",
        "complaint",
        "follow up"
    ],

    "email":[
        "Dear Manager...",
        "Dear Sir...",
        "Dear Customer Support...",
        "Dear Recruiter..."
    ]
}

df = pd.DataFrame(data)
df.head()

,instruction,email
0,promotion request,Dear Manager...
1,leave request,Dear Sir...
2,complaint,Dear Customer Support...
3,follow up,Dear Recruiter...


In [ ]:
import re

def clean_text(text):

  text = text.lower()

  text = text.lower()

  text =resub(
      r'[^a-zA-Z ]',
      '',
      text
  )

  return text



In [ ]:
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = stopwords.words('english')

print(stop_words[:20])

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

print(
    stemmer.stem("running")
)

run


In [ ]:
import nltk
nltk.download('wordnet')

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

print(
    lemmatizer.lemmatize("running")
)

[nltk_data] Downloading package wordnet to /root/nltk_data...


running


In [ ]:
{
    "instruction":
    "write follow up email",
    "response":
    "Dear Recruiter..."
}


{'instruction': 'write follow up email', 'response': 'Dear Recruiter...'}

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

dataset

Dataset({
    features: ['instruction', 'email'],
    num_rows: 4
})

In [ ]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [ ]:
def preprocess(example):

  model_inputs = tokenizer(
      example["instruction"],
      truncation=True,
      padding="max_length",
      max_length=128
  )

  labels = tokenizer(
      example["email"],
      truncation=True,
      padding="max_length",
      max_length=512,
  )

  model_inputs['labels'] = labels['input_ids']

  return model_inputs

In [ ]:
tokenized_dataset = dataset.map(
    preprocess
)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [ ]:
from transformers import (
    TrainingArguments,
    Trainer,
)

training_args = TrainingArguments(
    output_dir="./email_models",

    num_train_epochs=3,

    logging_steps=10,

    save_steps=50
)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

print("model Loaded Successfully")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model Loaded Successfully


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=35.71372985839844, metrics={'train_runtime': 366.7997, 'train_samples_per_second': 0.033, 'train_steps_per_second': 0.008, 'total_flos': 2054272057344.0, 'train_loss': 35.71372985839844, 'epoch': 3.0})

In [ ]:
trainer.save_model(
    "email_writer_model"
 )

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    "email_writer_model"
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
